# 08 – Bronze → Silver → Gold with Spark (Iceberg tables in the pile)

**Goal (Task 4, part 2):**
1. From **bronze** build cleaned Iceberg tables in **silver** (≥ 2 transformations per table), registered in the pile catalog.
2. Add records in the app → incremental **bronze** load → incremental **bronze → silver** processing → **time travel** before/after.
3. From **silver** build **two gold tables** (with **joins**), each with a **partition spec** that fits the expected queries, and run example queries.

**Who does what:** **Spark** does all the processing (cleaning, joins, aggregations). **PyIceberg** reads and writes the Iceberg tables in the **same SQLite catalog** as Task 3 (`read_iceberg` / `write_iceberg` in `spark_pile.py`). Data moves between them as Arrow tables.

| Layer | Namespace | Tables |
|---|---|---|
| 🥉 bronze | `bronze` | raw copies from Task 3 (`postgres_*`, `dogceo_breeds`) |
| 🥈 silver | `silver` | `shelters`, `adopters`, `animals`, `applications`, `dog_breeds` |
| 🥇 gold | `gold` | `shelter_monthly_activity`, `breed_popularity` |

In [ ]:
import pandas as pd
import sqlalchemy as sa
from pyspark.sql import functions as F, Window
from pyiceberg.expressions import AlwaysTrue, GreaterThan, GreaterThanOrEqual, LessThan, And, EqualTo
from pyiceberg.transforms import MonthTransform, IdentityTransform

from lake import POSTGRES_URL
from pile import run_bronze_incremental, show_catalog_db
from spark_pile import get_spark, read_iceberg, write_iceberg, get_catalog, snapshots

spark = get_spark("08-medallion")
catalog = get_catalog("silo")
for ns in ("silver", "gold"):
    catalog.create_namespace_if_not_exists(ns)
print("Namespaces:", catalog.list_namespaces())

## Part A – Bronze → Silver

### The cleaning rules

Every silver table gets:
- **Deduplication** → bronze is append-only, so an edited row exists in several versions. `latest_per_id` keeps only the **newest version per id** (window function `row_number()` over `updated_at`).
- **≥ 2 more transformations** per table (see the comments).
- **Technical dlt columns removed**, and two new ones: `_bronze_load_id` (which bronze load the row came from) and `_processed_at`.

In [ ]:
def latest_per_id(df):
    """Keep only the newest version of every row (bronze keeps all versions)."""
    newest_first = Window.partitionBy("id").orderBy(F.col("updated_at").desc(), F.col("_dlt_load_id").desc())
    return df.withColumn("_rn", F.row_number().over(newest_first)).filter("_rn = 1").drop("_rn")


def clean_shelters(df):
    return (latest_per_id(df)
            .withColumn("name", F.initcap(F.trim("name")))                        # 1. tidy text
            .withColumn("city", F.initcap(F.trim("city")))
            .withColumn("email", F.lower(F.trim("email")))                        # 2. emails in lower case
            .withColumn("capacity_band", F.when(F.col("capacity") < 50, "small")  # 3. new category column
                        .when(F.col("capacity") < 150, "medium").otherwise("large")))


def clean_adopters(df):
    return (latest_per_id(df)
            .withColumn("full_name", F.concat_ws(" ", F.initcap(F.trim("first_name")),   # 1. combine names
                                                      F.initcap(F.trim("last_name"))))
            .withColumn("email", F.lower(F.trim("email")))                               # 2. normalise email
            .withColumn("email_domain", F.split("email", "@").getItem(1))                # 3. extract domain
            .withColumn("city", F.initcap(F.trim("city")))
            .withColumn("birth_year", F.year("date_of_birth")))                          # 4. derived column


def clean_animals(df):
    return (latest_per_id(df)
            .withColumn("name", F.initcap(F.trim("name")))                                 # 1. tidy text
            .withColumn("species", F.lower(F.trim("species")))
            .withColumn("breed", F.coalesce(F.initcap(F.trim("breed")), F.lit("Unknown"))) # 2. fill missing breed
            .withColumn("sex", F.when(F.col("sex") == "M", "male")                         # 3. decode M/F
                        .when(F.col("sex") == "F", "female").otherwise("unknown"))
            .withColumn("age_years", F.col("age_years").cast("double"))                    # 4. decimal -> double
            .withColumn("age_group", F.when(F.col("age_years") < 1, "baby")                # 5. age groups
                        .when(F.col("age_years") < 3, "young")
                        .when(F.col("age_years") < 8, "adult").otherwise("senior"))
            .withColumn("found_home", F.col("status").isin("reserved", "adopted")))        # 6. label for ML


def clean_applications(df):
    return (latest_per_id(df)
            .withColumn("status", F.lower(F.trim("status")))
            .withColumn("days_to_decision", F.datediff("decided_at", "submitted_at"))      # 1. duration
            .withColumn("is_successful", F.col("status").isin("approved", "adopted"))      # 2. flag
            .withColumn("submitted_month", F.trunc("submitted_at", "month"))               # 3. month for reports
            .drop("message"))                                                              # 4. drop free text


SILVER = {"shelters": clean_shelters, "adopters": clean_adopters,
          "animals": clean_animals, "applications": clean_applications}
DLT_COLUMNS = ["_dlt_load_id", "_dlt_id", "_load_id", "_source_resource", "_ingestion_type"]

### Incremental processing

`refresh_silver()` only processes **bronze rows from loads that silver hasn't seen yet**:
1. find the last processed load: `max(_bronze_load_id)` in the silver table
2. read from bronze only rows with `_dlt_load_id` **greater** than that (an Iceberg filter, pushed down to the scan)
3. clean them with Spark
4. **upsert** (merge) into silver by `id`: existing rows are **updated**, new rows are **inserted**

The first run processes everything, because silver is still empty.

In [ ]:
def last_processed_load(identifier):
    if not catalog.table_exists(identifier):
        return None
    loads = catalog.load_table(identifier).scan(selected_fields=("_bronze_load_id",)).to_arrow().column(0)
    return max(loads.to_pylist()) if len(loads) else None


def refresh_silver(names=SILVER):
    for name in names:
        target = f"silver.{name}"
        last = last_processed_load(target)
        new_rows = GreaterThan("_dlt_load_id", last) if last else AlwaysTrue()
        bronze = read_iceberg(spark, f"bronze.postgres_{name}", row_filter=new_rows)
        print(f"{name:13} {bronze.count():5} new bronze rows (after load {last})")
        silver = (SILVER[name](bronze)
                  .withColumn("_bronze_load_id", F.col("_dlt_load_id"))
                  .withColumn("_processed_at", F.current_timestamp())
                  .drop(*DLT_COLUMNS))
        write_iceberg(silver, target, mode="upsert", join_cols=["id"])


refresh_silver()

### silver.dog_breeds (from the API)

The API table in bronze is a full `replace`, so silver is simply **overwritten**. Transformations: a nice display name, a **`breed_key`** to match breeds written in a different word order ("French Bulldog" ↔ "bulldog french"), duplicates removed, technical columns dropped.

In [ ]:
breeds = (read_iceberg(spark, "bronze.dogceo_breeds")
          .withColumn("display_name", F.initcap(F.trim("full_name")))
          .withColumn("breed_key", F.array_join(F.array_sort(F.split(F.lower(F.trim("full_name")), " ")), " "))
          .dropDuplicates(["breed_key"])
          .select("breed", "sub_breed", "display_name", "breed_key", "_source_system", "_ingested_at"))
write_iceberg(breeds, "silver.dog_breeds", mode="overwrite")
print("silver.dog_breeds:", breeds.count(), "breeds")

### Check silver: registered in the catalog, row counts, before vs after cleaning

In [ ]:
show_catalog_db("silo")

In [ ]:
pd.DataFrame([{"table": f"silver.{n}",
               "bronze_rows (all versions)": len(catalog.load_table(f"bronze.postgres_{n}").scan().to_arrow()),
               "silver_rows (latest per id)": len(catalog.load_table(f"silver.{n}").scan().to_arrow())}
              for n in SILVER])

In [ ]:
cols = ["id", "name", "species", "breed", "sex", "age_years"]
print("BRONZE (raw):")
read_iceberg(spark, "bronze.postgres_animals").select(*cols).orderBy("id").show(5)
print("SILVER (cleaned):")
read_iceberg(spark, "silver.animals").select(*cols, "age_group", "found_home").orderBy("id").show(5)

## Part B – New records → incremental bronze → incremental silver → time travel

First remember the current snapshot of `silver.animals`:

In [ ]:
snapshot_before = catalog.load_table("silver.animals").current_snapshot().snapshot_id
print("silver.animals snapshot BEFORE:", snapshot_before)

**Now change some data.**
- **Option A (for the presentation):** in the app http://localhost:5173 add 2–3 animals and edit one, then **skip** the next cell.
- **Option B:** run the next cell (inserts 2 animals and renames one with SQL).

In [ ]:
# Option B – skip if you used the app
engine = sa.create_engine(POSTGRES_URL)
with engine.begin() as conn:
    new_ids = conn.execute(sa.text("""
        INSERT INTO animals (shelter_id, name, species, breed, sex, age_years, size, arrived_at)
        VALUES (1, '  silver test 1 ', 'dog', 'french bulldog', 'F', 2, 'small', CURRENT_DATE),
               (2, 'silver test 2', 'cat', NULL, 'M', 0.5, 'small', CURRENT_DATE)
        RETURNING id""")).scalars().all()
    updated_id = conn.execute(sa.text(
        "UPDATE animals SET name = name || ' (renamed)' WHERE id = (SELECT min(id) FROM animals) RETURNING id")).scalar()
print("inserted:", new_ids, "| renamed:", updated_id)

**1) Incremental load PostgreSQL → bronze** (the Task 3 dlt pipeline, same `updated_at` cursor):

In [ ]:
run_bronze_incremental("silo")

**2) Incremental processing bronze → silver** (only the new bronze rows; run it twice → the second time there's nothing new):

In [ ]:
refresh_silver()
refresh_silver()

### Snapshots and time travel on silver.animals

An upsert creates an **overwrite** snapshot (rewrites the file that held the updated rows, with their new version left out) and **append** snapshots (the new versions and the new rows). The total grows only by the inserted rows.

In [ ]:
silver_animals = catalog.load_table("silver.animals")
snapshots(silver_animals)

In [ ]:
before = read_iceberg(spark, "silver.animals", snapshot_id=snapshot_before)
after = read_iceberg(spark, "silver.animals")
print(f"BEFORE (snapshot {snapshot_before}): {before.count()} animals")
print(f"AFTER  (snapshot {silver_animals.current_snapshot().snapshot_id}): {after.count()} animals")

print("\nNEW animals (only in AFTER) – note the cleaned name, breed and sex:")
after.join(before.select("id"), "id", "left_anti").select("id", "name", "species", "breed", "sex", "age_group").show()

print("CHANGED animals – the same id in both versions:")
changed = (after.alias("a").join(before.alias("b"), "id")
           .filter(F.col("a.name") != F.col("b.name"))
           .select("id", F.col("b.name").alias("name_BEFORE"), F.col("a.name").alias("name_AFTER")))
changed.show(truncate=False)

## Part C – Silver → Gold

### gold.shelter_monthly_activity

**Question it answers:** *How many applications and adoptions did each shelter have per month, and how fast were decisions?*
**Joins:** `applications` ⋈ `animals` (to get the shelter) ⋈ `shelters` (name, city).
**Partition spec:** `month(month_start)`. Reports are **per month or per period** ("October report", "last 12 months"), so a query reads only the partitions of those months.

In [ ]:
apps = read_iceberg(spark, "silver.applications")
animals = read_iceberg(spark, "silver.animals")
shelters = read_iceberg(spark, "silver.shelters")

monthly = (apps
           .join(animals.select(F.col("id").alias("animal_id"), "shelter_id"), "animal_id")
           .join(shelters.select(F.col("id").alias("shelter_id"), F.col("name").alias("shelter_name"), "city"),
                 "shelter_id")
           .groupBy("shelter_id", "shelter_name", "city", F.col("submitted_month").alias("month_start"))
           .agg(F.count("*").alias("applications"),
                F.sum(F.col("is_successful").cast("int")).alias("successful"),
                F.sum((F.col("status") == "adopted").cast("int")).alias("adoptions"),
                F.sum((F.col("status") == "rejected").cast("int")).alias("rejected"),
                F.round(F.avg("days_to_decision"), 1).alias("avg_days_to_decision"))
           .withColumn("success_rate", F.round(F.col("successful") / F.col("applications"), 3))
           .withColumn("_built_at", F.current_timestamp()))

gold_monthly = write_iceberg(monthly, "gold.shelter_monthly_activity", mode="overwrite",
                             partition_by=[("month_start", MonthTransform(), "month")])
print(gold_monthly.spec())
monthly.orderBy(F.desc("month_start"), F.desc("applications")).show(5)

### gold.breed_popularity

**Question it answers:** *Which breeds find a home most often, how many applications do they get, and how long do they stay?*
**Joins:** `animals` ⟕ (applications counted per animal) ⟕ `dog_breeds` (is the breed known in the Dog CEO API?). This combines **two sources**: our database and the API.
**Partition spec:** `identity(species)`. Users always look at **one species** ("top dog breeds", "cats that stay longest").

In [ ]:
applications_per_animal = apps.groupBy(F.col("animal_id").alias("id")).agg(F.count("*").alias("n_applications"))
dog_breeds = read_iceberg(spark, "silver.dog_breeds").select("breed_key", F.lit(True).alias("in_dog_api"))

breed_stats = (animals
               .join(applications_per_animal, "id", "left").fillna({"n_applications": 0})
               .withColumn("days_in_shelter", F.datediff(F.current_date(), "arrived_at"))
               .withColumn("breed_key", F.array_join(F.array_sort(F.split(F.lower("breed"), " ")), " "))
               .join(dog_breeds, "breed_key", "left")
               .groupBy("species", "breed")
               .agg(F.count("*").alias("animals"),
                    F.sum(F.col("found_home").cast("int")).alias("found_home"),
                    F.round(F.avg("n_applications"), 2).alias("avg_applications"),
                    F.round(F.avg("age_years"), 1).alias("avg_age"),
                    F.round(F.avg("days_in_shelter")).cast("int").alias("avg_days_in_shelter"),
                    F.max(F.coalesce("in_dog_api", F.lit(False))).alias("known_in_dog_api"))
               .withColumn("found_home_rate", F.round(F.col("found_home") / F.col("animals"), 3))
               .withColumn("_built_at", F.current_timestamp()))

gold_breeds = write_iceberg(breed_stats, "gold.breed_popularity", mode="overwrite",
                            partition_by=[("species", IdentityTransform(), "species")])
print(gold_breeds.spec())
breed_stats.groupBy("species").agg(F.count("*").alias("breeds"), F.sum("animals").alias("animals")).show()

In [ ]:
show_catalog_db("silo")      # bronze + silver + gold, all in the same catalog

### Expected queries on gold

The `row_filter` is given to Iceberg, which uses the **partition spec** to skip whole files. `plan_files()` shows how many data files are really read.

In [ ]:
def files_read(table, row_filter):
    return f"{len(list(table.scan(row_filter=row_filter).plan_files()))} of {len(list(table.scan().plan_files()))} data files"

# Q1: the monthly report – best shelters in the last full month
last_month = monthly.agg(F.max("month_start")).first()[0]
report_month = (pd.Timestamp(last_month) - pd.offsets.MonthBegin(1)).date()
next_month = (pd.Timestamp(report_month) + pd.offsets.MonthBegin(1)).date()
month_filter = And(GreaterThanOrEqual("month_start", str(report_month)), LessThan("month_start", str(next_month)))

print(f"Q1 – top shelters in {report_month:%B %Y} → reads", files_read(gold_monthly, month_filter))
(read_iceberg(spark, "gold.shelter_monthly_activity", row_filter=month_filter)
 .select("shelter_name", "city", "applications", "adoptions", "success_rate", "avg_days_to_decision")
 .orderBy(F.desc("adoptions"), F.desc("applications")).show(5))

In [ ]:
# Q2: one shelter's trend over the last 12 months
year_ago = (pd.Timestamp(report_month) - pd.DateOffset(months=11)).date()
period = And(GreaterThanOrEqual("month_start", str(year_ago)), LessThan("month_start", str(next_month)))
print(f"Q2 – trend of shelter 1 from {year_ago} → reads", files_read(gold_monthly, period))
(read_iceberg(spark, "gold.shelter_monthly_activity", row_filter=And(period, EqualTo("shelter_id", 1)))
 .select("month_start", "applications", "adoptions", "success_rate").orderBy("month_start").show(12))

In [ ]:
# Q3: top dog breeds that find a home (at least 5 animals), with Spark SQL
print("Q3 – reads", files_read(gold_breeds, EqualTo("species", "dog")))
read_iceberg(spark, "gold.breed_popularity", row_filter=EqualTo("species", "dog")).createOrReplaceTempView("dog_breeds_gold")
spark.sql("""
    SELECT breed, animals, found_home_rate, avg_applications, avg_days_in_shelter, known_in_dog_api
    FROM dog_breeds_gold
    WHERE animals >= 5
    ORDER BY found_home_rate DESC, animals DESC
    LIMIT 10
""").show(truncate=False)

In [ ]:
# Q4: cat breeds that stay longest in the shelter
print("Q4 – reads", files_read(gold_breeds, EqualTo("species", "cat")))
(read_iceberg(spark, "gold.breed_popularity", row_filter=EqualTo("species", "cat"))
 .filter("animals >= 5").orderBy(F.desc("avg_days_in_shelter"))
 .select("breed", "animals", "avg_days_in_shelter", "found_home_rate").show(5, truncate=False))